In [ ]:
!pip install scispacy
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz
import nltk; nltk.download('punkt_tab')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 8.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.0/865.0 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 82.5 MB/s eta 0:00:00
  Created wheel for en_ner_bc5cdr_md: filename=en_ner_bc5cdr_md-0.5.4-py3-none-any.whl size=119787677 sha256=8a5d2365327c1dc335fcf30588b0e41a47b4432cb58c87a119cd8b3691a02737
  Stored in directory: /root/.cache/pip/wheels/40/f3/2b/51cee972ff42cbe21ddaf5abef7376bb35c2c2ca26a96220b8
Successfully built en_ner_bc5cdr_md
  Attempting uninstall: blis
    Found existing installation: blis 1.3.3
    Uninstalling blis-1.3.3:
      Successfully uninstalled blis-1.3.3
  Attempting uninstall

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
import os
import re
import time
import requests
import pandas as pd
import yaml
from bs4 import BeautifulSoup
import nltk



try:
    import scispacy
    import spacy
    nlp = spacy.load("en_ner_bc5cdr_md")
    NLP_BACKEND = "scispacy"
except Exception:
    nlp = None
    NLP_BACKEND = "regex"
    print("[WARN] scispacy not available – falling back to regex NER")

nltk.download("punkt_tab", quiet=True)



def load_config(path: str) -> dict:
    with open(path) as f:
        return yaml.safe_load(f)


def safe_get(url: str, params: dict = None, timeout: int = 20) -> requests.Response | None:

    for attempt in range(3):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            print(f"  [WARN] attempt {attempt+1}/3 failed for {url}: {e}")
            time.sleep(2 ** attempt)
    return None

# Regex fallback
_CHEM_RE = re.compile(
    r'\b(?:[A-Z]?[a-z]{2,}-?)?(?:acid|amine|ol\b|ine\b|ate\b|ose\b|'
    r'lipid|ceramide|sphingo\w+|phospho\w+|acyl\w*|sterol|'
    r'choline|carnitine|tryptophan|tyrosine|dopamine|serotonin|'
    r'metabolite|bile\s+acid|\d+:\d+)\w*',
    re.IGNORECASE,
)

def extract_chemicals_from_sentence(sentence: str) -> list[str]:
    if nlp is not None:
        doc = nlp(sentence)
        return [ent.text.lower() for ent in doc.ents if ent.label_ == "CHEMICAL"]
    else:
        return [m.group().lower() for m in _CHEM_RE.finditer(sentence)]



EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"


def search_pubmed(query: str, retmax: int = 200) -> list[str]:
    r = safe_get(f"{EUTILS}/esearch.fcgi", {
        "db": "pubmed", "term": query,
        "retmax": retmax, "retmode": "json",
    })
    if r is None:
        return []
    try:
        return r.json()["esearchresult"]["idlist"]
    except Exception as e:
        print(f"  [ERROR] PubMed search parse: {e}")
        return []


def fetch_abstracts_xml(pmids: list[str]) -> str:
    if not pmids:
        return ""

    r = safe_get(f"{EUTILS}/efetch.fcgi", {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml",
        "rettype": "abstract",
    })
    return r.text if r else ""


def parse_pubmed_xml(xml_text: str) -> list[dict]:
    soup = BeautifulSoup(xml_text, "lxml-xml")
    records = []
    for article in soup.find_all("PubmedArticle"):
        pmid_tag = article.find("PMID")
        pmid = pmid_tag.text if pmid_tag else "unknown"
        title_tag = article.find("ArticleTitle")
        title = title_tag.text if title_tag else ""
        abstract_parts = article.find_all("AbstractText")
        abstract = " ".join(a.text for a in abstract_parts)
        if abstract.strip():
            records.append({"pmid": pmid, "title": title, "abstract": abstract})
    return records

def fetch_pmc_fulltext(pmids: list[str]) -> str:

    r = safe_get(f"{EUTILS}/efetch.fcgi", {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml",
        "rettype": "full",
    })
    if not r:
        return ""
    soup = BeautifulSoup(r.text, "lxml-xml")
    body = soup.find("body")
    if not body:
        return ""
    for tag in body.find_all(["ref-list", "supplementary-material"]):
        tag.decompose()
    return body.get_text(separator=" ", strip=True)


def fulltext(records: list[dict]) -> list[dict]:

    pmid     = [r["pmid"] for r in records]

    for rec in records:
        if pmid:
            fulltext = fetch_pmc_fulltext(pmid)
            rec.update({"fulltext": fulltext, "pmid": pmid, "text_source": "fulltext"})
            time.sleep(0.5)
        else:
            rec.update({"fulltext": "", "pmid": None, "text_source": "abstract_only"})
    return records


def filter_records(records: list[dict], diseases: list[str], biofluids: list[str]) -> list[dict]:
    kept = []
    for rec in records:
        text = (rec["title"] + " " + rec["abstract"]).lower()
        if any(d in text for d in diseases) and any(b in text for b in biofluids):
            kept.append(rec)
    return kept


def extract_metabolites_pubmed(records: list[dict]) -> list[dict]:
    results = []
    for rec in records:
        sentences = nltk.sent_tokenize(rec["abstract"])
        for sent in sentences:
            chemicals = extract_chemicals_from_sentence(sent)
            for chem in chemicals:
                results.append({
                    "metabolite": chem,
                    "pmid": rec["pmid"],
                    "title": rec["title"],
                    "sentence": sent,
                    "source": "pubmed",
                })
    return results


def run_pubmed(config: dict, out_path: str) -> pd.DataFrame:
    all_results = []
    for query in config["queries"]:
        print(f"[PubMed] Query: {query!r}")
        pmids = search_pubmed(query, retmax=200)
        print(f"  Found {len(pmids)} PMIDs")
        if not pmids:
            continue
        xml = fetch_abstracts_xml(pmids)
        records = parse_pubmed_xml(xml)
        print(f"  Parsed {len(records)} abstracts")
        filtered = filter_records(records, config["diseases"], config["biofluids"])
        print(f"  After filter: {len(filtered)} abstracts")
        mets = extract_metabolites_pubmed(filtered)
        print(f"  Extracted {len(mets)} chemical mentions")
        all_results.extend(mets)
        time.sleep(0.4)

    df = pd.DataFrame(all_results)
    if not df.empty:
        df = df.drop_duplicates(subset=["metabolite", "pmid"])
    df.to_csv(out_path, index=False)
    print(f"[PubMed] Saved {len(df)} rows → {out_path}\n")
    return df


if __name__ == "__main__":


    CONFIG_PATH  = "/content/drive/MyDrive/DATA/config.yaml"
    OUT_DIR      = "/content/drive/MyDrive/DATA"


    os.makedirs(OUT_DIR, exist_ok=True)
    config = load_config(CONFIG_PATH)

    run_pubmed(
        config,
        out_path=f"{OUT_DIR}/pubmed_metabolites.csv",
    )

    print("Pipeline finished.")

/usr/local/lib/python3.12/dist-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


[PubMed] Query: 'human plasma LC-MS Parkinsons'
  Found 104 PMIDs
  Parsed 104 abstracts
  After filter: 32 abstracts
  Extracted 277 chemical mentions
[PubMed] Query: 'human serum LC-MS Parkinsons'
  Found 57 PMIDs
  Parsed 57 abstracts
  After filter: 20 abstracts
  Extracted 125 chemical mentions
[PubMed] Saved 202 rows → /content/drive/MyDrive/DATA/pubmed_metabolites.csv

Pipeline finished.


In [ ]:
import os
from google.colab import drive

# Unmount if already mounted and clean up any remaining files
if os.path.isdir('/content/drive') and os.path.ismount('/content/drive'):
    drive.flush_and_unmount()
if os.path.exists('/content/drive'):
    !rm -rf /content/drive/*

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


Version without pubchem:

In [ ]:
import os
import re
import time
import requests
import pandas as pd
import yaml
from bs4 import BeautifulSoup
from typing import Optional
import nltk

try:
    import scispacy
    import spacy
    nlp = spacy.load("en_ner_bc5cdr_md")
    NLP_BACKEND = "scispacy"
except Exception:
    nlp = None
    NLP_BACKEND = "regex"
    print("[WARN] scispacy not available – falling back to regex NER")

nltk.download("punkt_tab", quiet=True)
nltk.download("punkt", quiet=True)



def load_config(path: str) -> dict:
    with open(path) as f:
        return yaml.safe_load(f)



def safe_get(url: str, params: dict = None, timeout: int = 20) -> Optional[requests.Response]:
    for attempt in range(3):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            print(f"  [WARN] attempt {attempt+1}/3 failed for {url}: {e}")
            time.sleep(2 ** attempt)
    return None


_CHEM_RE = re.compile(
    r'\b(?:[A-Z]?[a-z]{2,}-?)?(?:acid|amine|ol\b|ine\b|ate\b|ose\b|'
    r'lipid|ceramide|sphingo\w+|phospho\w+|acyl\w*|sterol|'
    r'choline|carnitine|tryptophan|tyrosine|dopamine|serotonin|'
    r'metabolite|bile\s+acid|\d+:\d+)\w*',
    re.IGNORECASE,
)

def extract_chemicals_from_sentence(sentence: str) -> list:
    if nlp is not None:
        doc = nlp(sentence)
        return [ent.text.lower() for ent in doc.ents if ent.label_ == "CHEMICAL"]
    else:
        return [m.group().lower() for m in _CHEM_RE.finditer(sentence)]




EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"


def search_pubmed(query: str, retmax: int = 200) -> list:
    r = safe_get(f"{EUTILS}/esearch.fcgi", {
        "db": "pubmed", "term": query,
        "retmax": retmax, "retmode": "json",
    })
    if r is None:
        return []
    try:
        return r.json()["esearchresult"]["idlist"]
    except Exception as e:
        print(f"  [ERROR] PubMed search parse: {e}")
        return []


def fetch_abstracts_xml(pmids: list) -> str:
    if not pmids:
        return ""
    r = safe_get(f"{EUTILS}/efetch.fcgi", {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml",
        "rettype": "abstract",
    })
    return r.text if r else ""


def parse_pubmed_xml(xml_text: str) -> list:

    soup = BeautifulSoup(xml_text, "lxml-xml")
    records = []
    for article in soup.find_all("PubmedArticle"):
        pmid_tag = article.find("PMID")
        pmid = pmid_tag.text.strip() if pmid_tag else "unknown"
        title_tag = article.find("ArticleTitle")
        title = title_tag.get_text(" ", strip=True) if title_tag else ""
        abstract_parts = article.find_all("AbstractText")
        abstract = " ".join(a.get_text(" ", strip=True) for a in abstract_parts)
        # Always keep – full text may be available even when abstract isn't
        records.append({"pmid": pmid, "title": title, "abstract": abstract})
    return records


def pmid_to_pmcid(pmid: str) -> Optional[str]:

    r = safe_get(f"{EUTILS}/elink.fcgi", {
        "dbfrom": "pubmed",
        "db": "pmc",
        "id": pmid,
        "retmode": "json",
    })
    if not r:
        return None
    try:
        linksets = r.json().get("linksets", [])
        for ls in linksets:
            for ld in ls.get("linksetdbs", []):
                if ld.get("dbto") == "pmc":
                    ids = ld.get("links", [])
                    if ids:
                        return f"PMC{ids[0]}"
    except Exception as e:
        print(f"    [WARN] eLink parse error for PMID {pmid}: {e}")
    return None


def fetch_pmc_fulltext_xml(pmcid: str) -> str:
    """
    Fetch the full-text XML from PMC for a given PMC ID.
    Returns raw XML text or empty string.
    """
    numeric_id = pmcid.replace("PMC", "")
    r = safe_get(f"{EUTILS}/efetch.fcgi", {
        "db": "pmc",
        "id": numeric_id,
        "retmode": "xml",
        "rettype": "full",
    })
    return r.text if r else ""


def parse_pmc_fulltext_xml(xml_text: str) -> dict:

    soup = BeautifulSoup(xml_text, "lxml-xml")

    section_texts = {}
    body = soup.find("body")
    if body:
        for sec in body.find_all("sec"):
            title_tag = sec.find("title")
            sec_title = title_tag.get_text(" ", strip=True) if title_tag else "untitled"

            for nested in sec.find_all("sec"):
                nested.decompose()
            sec_text = sec.get_text(" ", strip=True)
            if sec_text:
                section_texts[sec_title] = sec_text

    body_text = " ".join(section_texts.values())

    tables = []
    for tbl in soup.find_all("table-wrap"):
        caption = tbl.find("caption")
        cap_text = caption.get_text(" ", strip=True) if caption else ""
        tbl_body = tbl.find("table")
        tbl_text = tbl_body.get_text(" ", strip=True) if tbl_body else ""
        if tbl_text:
            tables.append(f"[TABLE CAPTION: {cap_text}] {tbl_text}")


    supplementary = []
    for sup in soup.find_all("supplementary-material"):
        label = sup.find("label")
        label_text = label.get_text(" ", strip=True) if label else "Supplementary"
        sup_caption = sup.find("caption")
        cap_text = sup_caption.get_text(" ", strip=True) if sup_caption else ""

        inline_text = sup.get_text(" ", strip=True)
        if inline_text:
            supplementary.append(f"[SUPPLEMENTARY: {label_text} | {cap_text}] {inline_text}")

    for fig in soup.find_all("fig"):
        caption = fig.find("caption")
        if caption:
            supplementary.append(f"[FIGURE CAPTION] {caption.get_text(' ', strip=True)}")

    return {
        "body_text": body_text,
        "section_texts": section_texts,
        "supplementary": supplementary,
        "tables": tables,
    }


def enrich_with_fulltext(records: list) -> list:

    for rec in records:
        pmid = rec.get("pmid", "unknown")
        rec.setdefault("full_text", "")
        rec.setdefault("supplementary_text", "")
        rec.setdefault("table_text", "")
        rec.setdefault("section_texts", {})
        rec.setdefault("text_source", "abstract_only")

        if pmid == "unknown":
            continue

        print(f"  [fulltext] PMID {pmid} → resolving PMC ID …")
        pmcid = pmid_to_pmcid(pmid)
        time.sleep(0.34)

        if not pmcid:
            print(f"    No PMC full text available for PMID {pmid}")
            continue

        print(f"    → {pmcid} – fetching full text …")
        xml_text = fetch_pmc_fulltext_xml(pmcid)
        time.sleep(0.34)

        if not xml_text:
            print(f"    Empty XML returned for {pmcid}")
            continue

        parsed = parse_pmc_fulltext_xml(xml_text)

        rec["full_text"]          = parsed["body_text"]
        rec["section_texts"]      = parsed["section_texts"]
        rec["supplementary_text"] = " ".join(parsed["supplementary"])
        rec["table_text"]         = " ".join(parsed["tables"])
        rec["pmcid"]              = pmcid
        rec["text_source"]        = "fulltext"

    return records


def filter_records(records: list, diseases: list, biofluids: list) -> list:

    kept = []
    for rec in records:
        text = " ".join([
            rec.get("title", ""),
            rec.get("abstract", ""),
            rec.get("full_text", ""),
        ]).lower()
        if any(d in text for d in diseases) and any(b in text for b in biofluids):
            kept.append(rec)
    return kept



def extract_from_text_block(
    text: str,
    source_label: str,
    rec: dict,
) -> list:

    results = []
    if not text or not text.strip():
        return results

    try:
        sentences = nltk.sent_tokenize(text)
    except Exception:
        sentences = text.split(". ")

    for sent in sentences:
        chemicals = extract_chemicals_from_sentence(sent)
        if not chemicals:
            continue
        try:
            tokens = nltk.word_tokenize(sent)
        except Exception:
            tokens = sent.split()
        for chem in chemicals:
            results.append({
                "metabolite":        chem,
                "pmid":              rec.get("pmid", ""),
                "pmcid":             rec.get("pmcid", ""),
                "paper_title":       rec.get("title", ""),
                "abstract":          rec.get("abstract", ""),
                "sentence":          sent,
                "tokens":            tokens,
                "extraction_source": source_label,
                "text_source":       rec.get("text_source", "abstract_only"),
            })
    return results


def extract_metabolites(records: list) -> list:

    all_results = []
    for rec in records:

        section_texts = rec.get("section_texts", {})
        if section_texts:
            for sec_name, sec_text in section_texts.items():
                all_results.extend(
                    extract_from_text_block(sec_text, f"body:{sec_name}", rec)
                )
        elif rec.get("full_text"):

            all_results.extend(
                extract_from_text_block(rec["full_text"], "body", rec)
            )


        all_results.extend(
            extract_from_text_block(rec.get("table_text", ""), "table", rec)
        )


        all_results.extend(
            extract_from_text_block(rec.get("supplementary_text", ""), "supplementary", rec)
        )


        all_results.extend(
            extract_from_text_block(rec.get("abstract", ""), "abstract", rec)
        )

    return all_results



def run_pubmed(config: dict, out_path: str) -> pd.DataFrame:
    all_results = []

    for query in config["queries"]:
        print(f"\n{'='*60}")
        print(f"[PubMed] Query: {query!r}")
        pmids = search_pubmed(query, retmax=200)
        print(f"  Found {len(pmids)} PMIDs")
        if not pmids:
            continue

        xml     = fetch_abstracts_xml(pmids)
        records = parse_pubmed_xml(xml)
        print(f"  Parsed {len(records)} records")

        records = enrich_with_fulltext(records)

        filtered = filter_records(records, config["diseases"], config["biofluids"])
        print(f"  After disease/biofluid filter: {len(filtered)} records")
        if not filtered:
            continue

        mets = extract_metabolites(filtered)
        print(f"  Extracted {len(mets)} chemical mentions across all text layers")
        all_results.extend(mets)
        time.sleep(0.4)

    df = pd.DataFrame(all_results)

    if not df.empty:

        df = df.drop_duplicates(subset=["metabolite", "pmid", "sentence"])

        df["tokens"] = df["tokens"].apply(
            lambda t: " | ".join(t) if isinstance(t, list) else t
        )

        col_order = [
            "metabolite", "pmid", "pmcid", "paper_title",
            "abstract", "sentence", "tokens",
            "extraction_source", "text_source",
        ]
        df = df[[c for c in col_order if c in df.columns]]

    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"\n[PubMed] Saved {len(df)} rows → {out_path}")
    return df




if __name__ == "__main__":
    CONFIG_PATH = "/content/drive/MyDrive/DATA/config.yaml"
    OUT_DIR     = "/content/drive/MyDrive/DATA"

    os.makedirs(OUT_DIR, exist_ok=True)
    config = load_config(CONFIG_PATH)

    df = run_pubmed(
        config,
        out_path=f"{OUT_DIR}/pubmed_metabolites.csv",
    )

    print("\nPipeline finished.")
    print(df[["metabolite", "pmid", "paper_title", "extraction_source"]].head(20))

[WARN] scispacy not available – falling back to regex NER

[PubMed] Query: 'human plasma LC-MS Parkinsons'
  Found 107 PMIDs
  Parsed 107 records
  [fulltext] PMID 42168845 → resolving PMC ID …
    No PMC full text available for PMID 42168845
  [fulltext] PMID 42091637 → resolving PMC ID …
    No PMC full text available for PMID 42091637
  [fulltext] PMID 41940830 → resolving PMC ID …
    → PMC13110075 – fetching full text …
  [fulltext] PMID 41857309 → resolving PMC ID …
    No PMC full text available for PMID 41857309
  [fulltext] PMID 41814889 → resolving PMC ID …
    No PMC full text available for PMID 41814889
  [fulltext] PMID 41713120 → resolving PMC ID …
    No PMC full text available for PMID 41713120
  [fulltext] PMID 41633967 → resolving PMC ID …
    → PMC12868800 – fetching full text …
  [fulltext] PMID 41515898 → resolving PMC ID …
    → PMC12785599 – fetching full text …
  [fulltext] PMID 41092926 → resolving PMC ID …
    → PMC12535840 – fetching full text …
  [fulltext] 

In [ ]:


import time
import re
import requests
import pandas as pd
from pathlib import Path




INPUT_PATH  = "/content/drive/MyDrive/DATA/pubmed_metabolites.csv"
OUTPUT_DIR  = "/content/drive/MyDrive/DATA"
SLEEP_SEC   = 0.25

PUBCHEM_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"


def _get(url: str, params: dict = None, retries: int = 3) -> requests.Response | None:
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=15)
            if r.status_code == 404:
                return None          # not found – not an error worth retrying
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            wait = 2 ** attempt
            print(f"    [WARN] attempt {attempt+1}/{retries} failed: {e}  (retry in {wait}s)")
            time.sleep(wait)
    return None


def normalise(name: str) -> str:
    """Light normalisation before lookup – strip extra whitespace, lower-case."""
    return re.sub(r"\s+", " ", name.strip()).lower()




def lookup_by_name(name: str) -> dict:
    """
    Query PubChem /compound/name endpoint.
    Returns a dict with keys:
        found          bool
        cid            int | None
        iupac_name     str | None
        molecular_formula str | None
        molecular_weight  float | None
        canonical_smiles  str | None
        synonyms       list[str]   (top 5)
        pubchem_url    str | None
    """
    result = {
        "found": False,
        "cid": None,
        "iupac_name": None,
        "molecular_formula": None,
        "molecular_weight": None,
        "canonical_smiles": None,
        "synonyms": [],
        "pubchem_url": None,
    }

    # Step 1: resolve name → CID
    r = _get(f"{PUBCHEM_URL}/compound/name/{requests.utils.quote(name)}/cids/JSON")
    if r is None:
        return result

    try:
        cids = r.json()["IdentifierList"]["CID"]
    except (KeyError, ValueError):
        return result

    if not cids:
        return result

    cid = cids[0]
    result["cid"] = cid
    result["found"] = True
    result["pubchem_url"] = f"https://pubchem.ncbi.nlm.nih.gov/compound/{cid}"


    props = "IUPACName,MolecularFormula,MolecularWeight,CanonicalSMILES"
    r2 = _get(
        f"{PUBCHEM_URL}/compound/cid/{cid}/property/{props}/JSON"
    )
    if r2:
        try:
            p = r2.json()["PropertyTable"]["Properties"][0]
            result["iupac_name"]        = p.get("IUPACName")
            result["molecular_formula"] = p.get("MolecularFormula")
            result["molecular_weight"]  = p.get("MolecularWeight")
            result["canonical_smiles"]  = p.get("CanonicalSMILES")
        except (KeyError, IndexError):
            pass

    # Step 3: fetch top synonyms
    r3 = _get(f"{PUBCHEM_URL}/compound/cid/{cid}/synonyms/JSON")
    if r3:
        try:
            all_syn = r3.json()["InformationList"]["Information"][0]["Synonym"]
            result["synonyms"] = all_syn[:5]
        except (KeyError, IndexError):
            pass

    return result




def validate(input_path: str, output_dir: str = ".") -> None:
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)


    p = Path(input_path)
    if p.suffix.lower() in {".xlsx", ".xls"}:
        df = pd.read_excel(p)
    else:
        df = pd.read_csv(p)

    if "metabolite" not in df.columns:
        raise ValueError(
            f"Column 'metabolite' not found. Available columns: {list(df.columns)}"
        )


    unique_names = df["metabolite"].dropna().unique().tolist()
    print(f"\nValidating {len(unique_names)} unique metabolite names via PubChem …\n")

    cache: dict[str, dict] = {}
    for i, name in enumerate(unique_names, 1):
        norm = normalise(str(name))
        print(f"  [{i}/{len(unique_names)}] {name!r}", end="  →  ", flush=True)

        if norm in cache:
            print("(cached)")
            continue

        info = lookup_by_name(norm)
        cache[norm] = info
        status = f"CID {info['cid']}" if info["found"] else "NOT FOUND"
        print(status)
        time.sleep(SLEEP_SEC)


    summary_rows = []
    for name in unique_names:
        norm = normalise(str(name))
        info = cache.get(norm, {})
        summary_rows.append({
            "metabolite_raw":      name,
            "metabolite_norm":     norm,
            "pubchem_found":       info.get("found", False),
            "pubchem_cid":         info.get("cid"),
            "iupac_name":          info.get("iupac_name"),
            "molecular_formula":   info.get("molecular_formula"),
            "molecular_weight":    info.get("molecular_weight"),
            "canonical_smiles":    info.get("canonical_smiles"),
            "top_synonyms":        "; ".join(info.get("synonyms", [])),
            "pubchem_url":         info.get("pubchem_url"),
        })

    summary_df = pd.DataFrame(summary_rows)


    df["_norm"] = df["metabolite"].astype(str).apply(normalise)
    summary_df["_norm"] = summary_df["metabolite_norm"]
    validated_df = df.merge(
        summary_df.drop(columns=["metabolite_raw", "metabolite_norm"]),
        on="_norm",
        how="left",
    ).drop(columns=["_norm"])


    validated_path = out / "validated_metabolites.xlsx"
    summary_path   = out / "pubchem_summary.xlsx"

    with pd.ExcelWriter(str(validated_path), engine="openpyxl") as writer:
        validated_df.to_excel(writer, index=False, sheet_name="Validated")

    with pd.ExcelWriter(str(summary_path), engine="openpyxl") as writer:
        summary_df.drop(columns=["_norm"], errors="ignore").to_excel(
            writer, index=False, sheet_name="PubChem Summary"
        )


        confirmed = summary_df[summary_df["pubchem_found"] == True].drop(
            columns=["_norm"], errors="ignore"
        )
        confirmed.to_excel(writer, index=False, sheet_name="Confirmed Only")

    n_found   = summary_df["pubchem_found"].sum()
    n_total   = len(summary_df)
    n_rows    = len(validated_df)
    pct       = 100 * n_found / n_total if n_total else 0

    print(f"\n{'='*55}")
    print(f"  Unique metabolite names : {n_total}")
    print(f"  Confirmed by PubChem    : {n_found}  ({pct:.1f} %)")
    print(f"  Not found               : {n_total - n_found}")
    print(f"  Total rows in output    : {n_rows}")
    print(f"\n  → {validated_path}")
    print(f"  → {summary_path}")
    print(f"{'='*55}\n")




if __name__ == "__main__":
    validate(
        input_path  = INPUT_PATH,
        output_dir  = OUTPUT_DIR,

    )


Validating 884 unique metabolite names via PubChem …

  [1/884] 'hexosylceramides'  →  NOT FOUND
  [2/884] 'hexosylsphingosines'  →  NOT FOUND
  [3/884] 'glycosphingolipids'  →  NOT FOUND
  [4/884] 'lipid'  →  NOT FOUND
  [5/884] 'glucosylceramides'  →  NOT FOUND
  [6/884] 'galactosylceramides'  →  CID 52949283
  [7/884] 'glucosylsphingosine'  →  CID 5280570
  [8/884] 'galactosylsphingosine'  →  CID 5280458
  [9/884] 'ceramide'  →  CID 139583739
  [10/884] 'lactosylceramide'  →  CID 6450208
  [11/884] 'sphingosine'  →  CID 5280335
  [12/884] 'baseline'  →  NOT FOUND
  [13/884] 'sphingolipid'  →  NOT FOUND
  [14/884] 'demonstrate'  →  NOT FOUND
  [15/884] 'acidic'  →  NOT FOUND
  [16/884] 'lipids'  →  NOT FOUND
  [17/884] 'lipidomics'  →  NOT FOUND
  [18/884] 'phospholipids'  →  NOT FOUND
  [19/884] 'sphingolipids'  →  NOT FOUND
  [20/884] 'lipidome'  →  NOT FOUND
  [21/884] 'mediate'  →  NOT FOUND
  [22/884] 'evaluate'  →  NOT FOUND
  [23/884] 'examined'  →  NOT FOUND
  [24/884] 'dete